In [9]:
"""
CUB Frustration Search (Jupyter Notebook Version) — supports your nested tree.

Your layout (from tree.txt):
  ./CUB_200_2011/attributes.txt
  ./CUB_200_2011/CUB_200_2011/{images.txt, classes.txt, attributes/...}

This cell auto-detects both:
  (A) standard: ./CUB_200_2011/{images.txt, attributes/attributes.txt, ...}
  (B) nested:   ./CUB_200_2011/attributes.txt + ./CUB_200_2011/CUB_200_2011/{...}
"""

import os
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd

# Optional statsmodels for independent association testing
try:
    import statsmodels.api as sm
    HAS_SM = True
except Exception as e:
    HAS_SM = False
    _SM_ERR = str(e)

# ============================================================
# CONFIGURATION
# ============================================================

MIN_ABS_CORR = 0.30
MIN_DELTA = 0.15
ALPHA = 0.10
USE_TRAIN_ONLY = False
PAIRS_PER_TRIPLE = 25
ALPHA = 0.05
USE_TRAIN_ONLY = True
RANDOM_SEED = 0

RUN_NOW = True  # set False to just define functions

# ============================================================
# PATH RESOLUTION (auto-detect standard vs your nested layout)
# ============================================================

BASE = os.path.join(os.getcwd(), "CUB_200_2011")

# Standard layout candidates
std_root = BASE
std_attr_names = os.path.join(std_root, "attributes", "attributes.txt")

# Your nested layout candidates (per tree.txt)
nested_root = os.path.join(BASE, "CUB_200_2011")
nested_attr_names = os.path.join(BASE, "attributes.txt")

def _exists_all(paths):
    return all(os.path.exists(p) for p in paths)

# Decide which layout we have
if _exists_all([os.path.join(std_root, "images.txt"), std_attr_names]):
    CUB_ROOT = std_root
    ATTR_NAMES_PATH = std_attr_names
elif _exists_all([os.path.join(nested_root, "images.txt"), nested_attr_names]):
    CUB_ROOT = nested_root
    ATTR_NAMES_PATH = nested_attr_names
else:
    print("Current working directory:", os.getcwd())
    print("Listing:", os.listdir(os.getcwd())[:80])
    raise FileNotFoundError(
        "Could not auto-detect CUB layout.\n"
        "Expected either:\n"
        "  (A) ./CUB_200_2011/images.txt and ./CUB_200_2011/attributes/attributes.txt\n"
        "or (B) ./CUB_200_2011/attributes.txt and ./CUB_200_2011/CUB_200_2011/images.txt\n"
    )

print("✅ Using CUB_ROOT:", CUB_ROOT)
print("✅ Using ATTR_NAMES_PATH:", ATTR_NAMES_PATH)

# ============================================================
# DATA LOADING
# ============================================================

def load_metadata():
    def read_two_col(path, c1, c2):
        return pd.read_csv(path, sep=r"\s+", header=None, names=[c1, c2], engine="python")

    images = read_two_col(os.path.join(CUB_ROOT, "images.txt"), "image_id", "rel_path")
    labels = read_two_col(os.path.join(CUB_ROOT, "image_class_labels.txt"), "image_id", "class_id")
    split = read_two_col(os.path.join(CUB_ROOT, "train_test_split.txt"), "image_id", "is_train")
    classes = read_two_col(os.path.join(CUB_ROOT, "classes.txt"), "class_id", "class_name")

    # NOTE: attributes.txt location differs between layouts; we use ATTR_NAMES_PATH detected above
    attr_names = read_two_col(ATTR_NAMES_PATH, "attr_id", "attr_name")

    class_attr = pd.read_csv(
        os.path.join(CUB_ROOT, "attributes", "class_attribute_labels_continuous.txt"),
        sep=r"\s+", header=None, engine="python"
    )
    if class_attr.shape != (200, 312):
        print("Warning: expected class_attr to be (200,312) but got", class_attr.shape)

    img_attr_path = os.path.join(CUB_ROOT, "attributes", "image_attribute_labels.txt")
    if not os.path.exists(img_attr_path):
        raise FileNotFoundError(f"Missing image_attribute_labels.txt at {img_attr_path}")

    return images, labels, split, classes, attr_names, class_attr, img_attr_path

# ============================================================
# CORRELATION + FRUSTRATION
# ============================================================

def compute_corr_matrix(class_attr: pd.DataFrame) -> np.ndarray:
    X = class_attr.to_numpy(dtype=float)
    X = X - X.mean(axis=0, keepdims=True)
    std = X.std(axis=0, ddof=1)
    std[std == 0] = 1.0
    X /= std
    R = (X.T @ X) / (X.shape[0] - 1)
    np.fill_diagonal(R, 1.0)
    return np.clip(R, -1.0, 1.0)

@dataclass
class Triple:
    i: int
    j: int
    k: int
    r_ij: float
    r_jk: float
    r_ki: float
    product: float
    strength: float

def find_frustrated_triples(R: np.ndarray) -> List[Triple]:
    p = R.shape[0]
    triples: List[Triple] = []
    for i in range(p):
        for j in range(i + 1, p):
            rij = R[i, j]
            if abs(rij) < MIN_ABS_CORR:
                continue
            for k in range(j + 1, p):
                rik = R[i, k]
                if abs(rik) < MIN_ABS_CORR:
                    continue
                rjk = R[j, k]
                if abs(rjk) < MIN_ABS_CORR:
                    continue
                prod = rij * rjk * R[k, i]
                if prod < 0:
                    strength = min(abs(rij), abs(rjk), abs(R[k, i]))
                    triples.append(Triple(i, j, k, float(rij), float(rjk), float(R[k, i]), float(prod), float(strength)))

    triples.sort(key=lambda t: (t.strength, abs(t.product)), reverse=True)
    return triples[:MAX_TRIPLES]

# ============================================================
# CLASS PAIR SEARCH
# ============================================================

def propose_pairs(class_attr: pd.DataFrame, triple: Tuple[int, int, int]) -> List[Tuple[int, int]]:
    i, j, k = triple
    A = class_attr.to_numpy(dtype=float)

    d1 = np.abs(A[:, i][:, None] - A[:, i][None, :])
    d2 = np.abs(A[:, j][:, None] - A[:, j][None, :])
    d3 = np.abs(A[:, k][:, None] - A[:, k][None, :])

    mask = (d1 >= MIN_DELTA) & (d2 >= MIN_DELTA) & (d3 >= MIN_DELTA)
    iu, ju = np.triu_indices(200, k=1)
    valid = mask[iu, ju]
    if not np.any(valid):
        return []

    scores = (d1 + d2 + d3)[iu, ju][valid]
    iu = iu[valid]
    ju = ju[valid]

    idx = np.argsort(-scores)[:PAIRS_PER_TRIPLE]
    return [(int(iu[x] + 1), int(ju[x] + 1)) for x in idx]  # 1-based class IDs

# ============================================================
# INDEPENDENT ASSOCIATION TEST
# ============================================================

def validate_independence(
    pair: Tuple[int, int],
    triple: Tuple[int, int, int],
    labels: pd.DataFrame,
    split: pd.DataFrame,
    img_attr_path: str
):
    if not HAS_SM:
        return False, f"statsmodels not installed/importable: {_SM_ERR}"

    a, b = pair
    ai, aj, ak = [x + 1 for x in triple]  # 1-based attr ids in file

    df = labels.copy()
    df = df[df["class_id"].isin([a, b])]

    if USE_TRAIN_ONLY:
        df = df.merge(split, on="image_id", how="inner")
        df = df[df["is_train"] == 1]

    if df.empty:
        return False, "no images left after filtering"

    y = (df["class_id"] == b).astype(int).to_numpy()
    image_ids = set(df["image_id"].astype(int).tolist())

    rows = []
    for chunk in pd.read_csv(
        img_attr_path,
        sep=r"\s+",
        header=None,
        names=["image_id", "attr_id", "is_present", "certainty", "time"],
        chunksize=1_000_000,
        engine="python"
    ):
        chunk = chunk[chunk["image_id"].isin(image_ids)]
        chunk = chunk[chunk["attr_id"].isin([ai, aj, ak])]
        if not chunk.empty:
            rows.append(chunk)

    if not rows:
        return False, "no attribute rows found for this pair/triple"

    df_attr = pd.concat(rows, ignore_index=True)

    X = df_attr.pivot_table(
        index="image_id",
        columns="attr_id",
        values="is_present",
        aggfunc="max"
    ).fillna(0.0)

    for col in [ai, aj, ak]:
        if col not in X.columns:
            X[col] = 0.0
    X = X[[ai, aj, ak]]

    X = X.reindex(df["image_id"].astype(int).tolist()).fillna(0.0)

    if (X.std(axis=0) == 0).any():
        return False, "zero variance in at least one concept within this class pair"

    X_ = sm.add_constant(X, has_constant="add")
    try:
        model = sm.Logit(y, X_)
        res = model.fit(disp=False, maxiter=200)
    except Exception as e:
        return False, f"logit failed: {e}"

    pvals = res.pvalues.iloc[1:].to_numpy()  # exclude intercept
    ok = bool(np.all(np.isfinite(pvals)) and np.all(pvals < ALPHA))
    return ok, pvals.tolist()

# ============================================================
# RUN
# ============================================================

def run_search():
    images, labels, split, classes, attr_names, class_attr, img_attr_path = load_metadata()

    R = compute_corr_matrix(class_attr)
    triples = find_frustrated_triples(R)

    print("\n=== Frustrating Triples (top 10) ===\n")
    for t in triples[:10]:
        print(
            attr_names.iloc[t.i]["attr_name"], "|",
            attr_names.iloc[t.j]["attr_name"], "|",
            attr_names.iloc[t.k]["attr_name"],
            f"| product={t.product:.4f} strength={t.strength:.3f}"
        )

    print("\n=== Searching for Binary Tasks (first VALID hit per run) ===\n")
    for t in triples:
        triple = (t.i, t.j, t.k)
        pairs = propose_pairs(class_attr, triple)

        for pair in pairs:
            ok, info = validate_independence(pair, triple, labels, split, img_attr_path)
            if ok:
                a_name = classes.loc[classes.class_id == pair[0], "class_name"].values[0]
                b_name = classes.loc[classes.class_id == pair[1], "class_name"].values[0]
                print("\n✔ VALID TASK")
                print("Triple:",
                      attr_names.iloc[t.i]["attr_name"], "|",
                      attr_names.iloc[t.j]["attr_name"], "|",
                      attr_names.iloc[t.k]["attr_name"])
                print("Classes:", a_name, "vs", b_name)
                print("p-values:", [f"{p:.2e}" for p in info])
                return {"triple": triple, "pair": pair, "pvals": info, "triple_obj": t}

    print("\nNo fully validated tasks found under current thresholds.")
    return None

if RUN_NOW:
    result = run_search()

✅ Using CUB_ROOT: /home/cbanerji/jupyterlab/CUB_200_2011/CUB_200_2011
✅ Using ATTR_NAMES_PATH: /home/cbanerji/jupyterlab/CUB_200_2011/attributes.txt

=== Frustrating Triples (top 10) ===

has_wing_pattern::solid | has_wing_pattern::spotted | has_wing_pattern::multi-colored | product=-0.0575 strength=0.360
has_back_pattern::spotted | has_wing_pattern::solid | has_wing_pattern::multi-colored | product=-0.0599 strength=0.354
has_bill_shape::all-purpose | has_bill_shape::cone | has_shape::perching-like | product=-0.0792 strength=0.327
has_bill_shape::all-purpose | has_bill_shape::cone | has_size::medium_(9_-_16_in) | product=-0.0586 strength=0.327
has_bill_shape::all-purpose | has_bill_shape::cone | has_size::small_(5_-_9_in) | product=-0.0529 strength=0.327
has_upperparts_color::white | has_shape::gull-like | has_wing_pattern::solid | product=-0.0414 strength=0.315
has_bill_shape::all-purpose | has_bill_shape::cone | has_shape::gull-like | product=-0.0320 strength=0.311
has_bill_shape::al

In [10]:
"""
CUB group-vs-group task builder for a chosen frustrating triple
(Jupyter-friendly, matches your nested CUB tree)

What it does:
1) Auto-detects your CUB layout (the nested one from your tree.txt, or the standard one).
2) Loads class-level attribute probabilities.
3) Builds a group-vs-group binary task for the triple:
      has_upperparts_color::white
      has_shape::gull-like
      has_wing_pattern::solid
   using thresholds on gull-like to define groups.
4) Reports:
   - which classes are in each group (names + IDs)
   - number of images in each group (optionally train-only)
   - group means for the three attributes
   - group deltas
5) Optionally exports an image-level CSV with:
      image_id, rel_path, y, c_white, c_gull, c_solid

Notes:
- Uses class_attribute_labels_continuous.txt for group selection.
- Uses image_attribute_labels.txt for per-image concept labels in the exported CSV.

Run in a notebook cell.
"""

import os
import numpy as np
import pandas as pd

# ----------------------------
# CONFIG
# ----------------------------
TRIPLE = (
    "has_upperparts_color::white",
    "has_shape::gull-like",
    "has_wing_pattern::solid",
)

# Group definition thresholds (tweak these)
GULL_HIGH = 0.60   # classes with gull-like prob > this go to Group A
GULL_LOW  = 0.20   # classes with gull-like prob < this go to Group B

# Keep groups from being too huge / too tiny (optional)
MAX_CLASSES_PER_GROUP = None  # e.g. 25, or None for no cap

# Use only train split for counting images / export (recommended later: True)
USE_TRAIN_ONLY = False

# Export per-image dataset for the chosen groups
EXPORT_IMAGE_LEVEL_CSV = True
EXPORT_PATH = "cub_gull_vs_nongull_task.csv"

# ----------------------------
# PATH AUTO-DETECT (supports your nested tree)
# ----------------------------
BASE = os.path.join(os.getcwd(), "CUB_200_2011")

std_root = BASE
std_attr_names = os.path.join(std_root, "attributes", "attributes.txt")

nested_root = os.path.join(BASE, "CUB_200_2011")
nested_attr_names = os.path.join(BASE, "attributes.txt")

def _exists_all(paths):
    return all(os.path.exists(p) for p in paths)

if _exists_all([os.path.join(std_root, "images.txt"), std_attr_names]):
    CUB_ROOT = std_root
    ATTR_NAMES_PATH = std_attr_names
elif _exists_all([os.path.join(nested_root, "images.txt"), nested_attr_names]):
    CUB_ROOT = nested_root
    ATTR_NAMES_PATH = nested_attr_names
else:
    print("cwd:", os.getcwd())
    print("cwd listing:", os.listdir(os.getcwd())[:80])
    raise FileNotFoundError(
        "Could not auto-detect CUB layout.\n"
        "Expected either:\n"
        "  (A) ./CUB_200_2011/images.txt and ./CUB_200_2011/attributes/attributes.txt\n"
        "or (B) ./CUB_200_2011/attributes.txt and ./CUB_200_2011/CUB_200_2011/images.txt\n"
    )

print("✅ Using CUB_ROOT:", CUB_ROOT)
print("✅ Using ATTR_NAMES_PATH:", ATTR_NAMES_PATH)

# ----------------------------
# LOAD METADATA
# ----------------------------
def read_two_col(path, c1, c2):
    return pd.read_csv(path, sep=r"\s+", header=None, names=[c1, c2], engine="python")

images = read_two_col(os.path.join(CUB_ROOT, "images.txt"), "image_id", "rel_path")
labels = read_two_col(os.path.join(CUB_ROOT, "image_class_labels.txt"), "image_id", "class_id")
split  = read_two_col(os.path.join(CUB_ROOT, "train_test_split.txt"), "image_id", "is_train")
classes = read_two_col(os.path.join(CUB_ROOT, "classes.txt"), "class_id", "class_name")
attr_names = read_two_col(ATTR_NAMES_PATH, "attr_id", "attr_name")

class_attr = pd.read_csv(
    os.path.join(CUB_ROOT, "attributes", "class_attribute_labels_continuous.txt"),
    sep=r"\s+", header=None, engine="python"
)
if class_attr.shape != (200, 312):
    print("⚠️ Warning: expected class_attr (200,312), got", class_attr.shape)

img_attr_path = os.path.join(CUB_ROOT, "attributes", "image_attribute_labels.txt")
if not os.path.exists(img_attr_path):
    raise FileNotFoundError(f"Missing: {img_attr_path}")

# ----------------------------
# FIND TRIPLE ATTRIBUTE INDICES
# ----------------------------
def get_attr_col_index(attr_name: str) -> int:
    """Return 0-based column index for class_attr given exact attr_name."""
    row = attr_names[attr_names["attr_name"] == attr_name]
    if row.empty:
        # give helpful partial matches
        matches = attr_names[attr_names["attr_name"].str.contains(attr_name.split("::")[-1], regex=False)]
        raise ValueError(f"Attribute not found: {attr_name}\nPartial matches:\n{matches.head(20)}")
    attr_id_1based = int(row.iloc[0]["attr_id"])
    return attr_id_1based - 1

white_idx = get_attr_col_index(TRIPLE[0])
gull_idx  = get_attr_col_index(TRIPLE[1])
solid_idx = get_attr_col_index(TRIPLE[2])

A = class_attr.to_numpy(dtype=float)
white = A[:, white_idx]
gull  = A[:, gull_idx]
solid = A[:, solid_idx]

# ----------------------------
# BUILD GROUPS (class indices 0..199)
# ----------------------------
gull_group = np.where(gull > GULL_HIGH)[0]
nongull_group = np.where(gull < GULL_LOW)[0]

# Optional caps: keep the most "confident" classes in each group
def cap_group(idx: np.ndarray, score: np.ndarray, max_n: int | None, descending=True) -> np.ndarray:
    if max_n is None or len(idx) <= max_n:
        return idx
    order = np.argsort(score[idx])
    if descending:
        order = order[::-1]
    return idx[order[:max_n]]

gull_group = cap_group(gull_group, gull, MAX_CLASSES_PER_GROUP, descending=True)
nongull_group = cap_group(nongull_group, gull, MAX_CLASSES_PER_GROUP, descending=False)

# ----------------------------
# REPORT GROUPS
# ----------------------------
def class_id_to_name(cid_1based: int) -> str:
    row = classes.loc[classes["class_id"] == cid_1based, "class_name"]
    return str(row.values[0]) if len(row) else str(cid_1based)

def describe_group(group_idx: np.ndarray, title: str):
    class_ids = (group_idx + 1).tolist()
    names = [class_id_to_name(cid) for cid in class_ids]
    df = pd.DataFrame({"class_id": class_ids, "class_name": names,
                       "p_white": white[group_idx], "p_gull": gull[group_idx], "p_solid": solid[group_idx]})
    df = df.sort_values("p_gull", ascending=(title.lower().find("non") >= 0)).reset_index(drop=True)
    print(f"\n=== {title} ===")
    print("n_classes:", len(df))
    return df

df_gulls = describe_group(gull_group, "Group A (Gull-like)")
df_nongulls = describe_group(nongull_group, "Group B (Non-gull-like)")

display(df_gulls.head(30))
display(df_nongulls.head(30))

def group_means(idx: np.ndarray):
    return float(white[idx].mean()), float(gull[idx].mean()), float(solid[idx].mean())

mA = group_means(gull_group)
mB = group_means(nongull_group)

print("\n=== Group means (class-level probabilities) ===")
print("A means [white, gull, solid]:", tuple(round(x, 3) for x in mA))
print("B means [white, gull, solid]:", tuple(round(x, 3) for x in mB))
print("Delta (A-B):", tuple(round(a-b, 3) for a, b in zip(mA, mB)))

# ----------------------------
# IMAGE COUNTS PER GROUP
# ----------------------------
def count_images_for_class_ids(class_ids_1based: list[int], use_train_only: bool) -> int:
    df = labels[labels["class_id"].isin(class_ids_1based)].copy()
    if use_train_only:
        df = df.merge(split, on="image_id", how="inner")
        df = df[df["is_train"] == 1]
    return int(df.shape[0])

A_class_ids = (gull_group + 1).tolist()
B_class_ids = (nongull_group + 1).tolist()

nA = count_images_for_class_ids(A_class_ids, USE_TRAIN_ONLY)
nB = count_images_for_class_ids(B_class_ids, USE_TRAIN_ONLY)

print("\n=== Image counts ===")
print("USE_TRAIN_ONLY:", USE_TRAIN_ONLY)
print("Group A images:", nA)
print("Group B images:", nB)
print("Imbalance ratio max/min:", round(max(nA, nB) / max(1, min(nA, nB)), 3))

# ----------------------------
# OPTIONAL: EXPORT IMAGE-LEVEL DATASET (y, concepts)
# ----------------------------
if EXPORT_IMAGE_LEVEL_CSV:
    # Build image list for both groups
    dfA = labels[labels["class_id"].isin(A_class_ids)].copy()
    dfB = labels[labels["class_id"].isin(B_class_ids)].copy()
    dfA["y"] = 1
    dfB["y"] = 0
    dfY = pd.concat([dfA, dfB], ignore_index=True)

    if USE_TRAIN_ONLY:
        dfY = dfY.merge(split, on="image_id", how="inner")
        dfY = dfY[dfY["is_train"] == 1].copy()

    dfY = dfY.merge(images, on="image_id", how="left")
    keep_image_ids = set(dfY["image_id"].astype(int).tolist())

    # Attribute IDs in file are 1-based
    white_id = white_idx + 1
    gull_id  = gull_idx + 1
    solid_id = solid_idx + 1
    wanted_attr_ids = {white_id, gull_id, solid_id}

    rows = []
    for chunk in pd.read_csv(
        img_attr_path,
        sep=r"\s+",
        header=None,
        names=["image_id", "attr_id", "is_present", "certainty", "time"],
        chunksize=1_000_000,
        engine="python",
    ):
        chunk = chunk[chunk["image_id"].isin(keep_image_ids)]
        chunk = chunk[chunk["attr_id"].isin(wanted_attr_ids)]
        if not chunk.empty:
            rows.append(chunk)

    if not rows:
        raise RuntimeError("No attribute rows found when exporting image-level CSV (check paths).")

    df_attr = pd.concat(rows, ignore_index=True)

    X = df_attr.pivot_table(
        index="image_id",
        columns="attr_id",
        values="is_present",
        aggfunc="max"
    ).fillna(0.0)

    # ensure columns exist
    for col in [white_id, gull_id, solid_id]:
        if col not in X.columns:
            X[col] = 0.0
    X = X[[white_id, gull_id, solid_id]].copy()
    X.columns = ["c_white", "c_gull", "c_solid"]

    out = dfY[["image_id", "rel_path", "y"]].copy()
    out = out.merge(X, left_on="image_id", right_index=True, how="left").fillna(0.0)

    out.to_csv(EXPORT_PATH, index=False)
    print(f"\n✅ Exported image-level dataset to: {EXPORT_PATH}")
    display(out.head(10))

✅ Using CUB_ROOT: /home/cbanerji/jupyterlab/CUB_200_2011/CUB_200_2011
✅ Using ATTR_NAMES_PATH: /home/cbanerji/jupyterlab/CUB_200_2011/attributes.txt

=== Group A (Gull-like) ===
n_classes: 131

=== Group B (Non-gull-like) ===
n_classes: 69


,class_id,class_name,p_white,p_gull,p_solid
0,66,066.Western_Gull,76.729560,74.842767,19.178082
1,62,062.Herring_Gull,73.943662,65.384615,51.048951
2,64,064.Ring_billed_Gull,60.740741,63.698630,53.543307
3,145,145.Elegant_Tern,68.217054,51.612903,47.286822
4,60,060.Glaucous_winged_Gull,62.318841,50.684932,40.000000
5,63,063.Ivory_Gull,96.103896,50.000000,73.291925
6,45,045.Northern_Fulmar,55.147059,50.000000,37.681159
7,146,146.Forsters_Tern,71.631206,49.350649,77.142857
8,44,044.Frigatebird,4.672897,48.872180,78.787879
9,144,144.Common_Tern,67.153285,47.019868,51.428571


,class_id,class_name,p_white,p_gull,p_solid
0,10,010.Red_winged_Blackbird,8.666667,0.0,15.231788
1,139,139.Scarlet_Tanager,1.481481,0.0,66.428571
2,150,150.Sage_Thrasher,30.612245,0.0,14.393939
3,159,159.Black_and_white_Warbler,88.311688,0.0,0.000000
4,161,161.Blue_winged_Warbler,28.333333,0.0,17.796610
5,167,167.Hooded_Warbler,3.496503,0.0,38.235294
6,168,168.Kentucky_Warbler,6.122449,0.0,33.333333
7,172,172.Nashville_Warbler,3.759398,0.0,29.133858
8,173,173.Orange_crowned_Warbler,2.307692,0.0,33.600000
9,174,174.Palm_Warbler,4.964539,0.0,28.467153



=== Group means (class-level probabilities) ===
A means [white, gull, solid]: (28.101, 10.685, 35.557)
B means [white, gull, solid]: (24.728, 0.0, 27.149)
Delta (A-B): (3.373, 10.685, 8.408)

=== Image counts ===
USE_TRAIN_ONLY: False
Group A images: 7704
Group B images: 4084
Imbalance ratio max/min: 1.886


ParserError: Expected 5 fields in line 709498, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

In [13]:
# ============================================================
# Gull vs Tern Group Analysis for Frustration Triple
# ============================================================

import numpy as np
import pandas as pd
import os
from scipy.stats import pointbiserialr

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

TRIPLE = (
    "has_upperparts_color::white",
    "has_shape::gull-like",
    "has_wing_pattern::solid",
)

USE_TRAIN_ONLY = False  # change later if desired

# ------------------------------------------------------------
# Resolve paths (same logic as before)
# ------------------------------------------------------------

BASE = os.path.join(os.getcwd(), "CUB_200_2011")

std_root = BASE
std_attr_names = os.path.join(std_root, "attributes", "attributes.txt")

nested_root = os.path.join(BASE, "CUB_200_2011")
nested_attr_names = os.path.join(BASE, "attributes.txt")

def _exists_all(paths):
    return all(os.path.exists(p) for p in paths)

if _exists_all([os.path.join(std_root, "images.txt"), std_attr_names]):
    CUB_ROOT = std_root
    ATTR_NAMES_PATH = std_attr_names
elif _exists_all([os.path.join(nested_root, "images.txt"), nested_attr_names]):
    CUB_ROOT = nested_root
    ATTR_NAMES_PATH = nested_attr_names
else:
    raise FileNotFoundError("Could not auto-detect CUB layout.")

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

def read_two_col(path, c1, c2):
    return pd.read_csv(path, sep=r"\s+", header=None, names=[c1, c2], engine="python")

images = read_two_col(os.path.join(CUB_ROOT, "images.txt"), "image_id", "rel_path")
labels = read_two_col(os.path.join(CUB_ROOT, "image_class_labels.txt"), "image_id", "class_id")
split  = read_two_col(os.path.join(CUB_ROOT, "train_test_split.txt"), "image_id", "is_train")
classes = read_two_col(os.path.join(CUB_ROOT, "classes.txt"), "class_id", "class_name")
attr_names = read_two_col(ATTR_NAMES_PATH, "attr_id", "attr_name")

class_attr = pd.read_csv(
    os.path.join(CUB_ROOT, "attributes", "class_attribute_labels_continuous.txt"),
    sep=r"\s+", header=None, engine="python"
)

img_attr_path = os.path.join(CUB_ROOT, "attributes", "image_attribute_labels.txt")

# ------------------------------------------------------------
# Get attribute indices
# ------------------------------------------------------------

def get_attr_index(name):
    row = attr_names[attr_names["attr_name"] == name]
    if row.empty:
        raise ValueError(f"Attribute not found: {name}")
    return int(row.iloc[0]["attr_id"]) - 1

white_idx = get_attr_index(TRIPLE[0])
gull_idx  = get_attr_index(TRIPLE[1])
solid_idx = get_attr_index(TRIPLE[2])

A = class_attr.to_numpy()
white = A[:, white_idx]
gull  = A[:, gull_idx]
solid = A[:, solid_idx]

# ------------------------------------------------------------
# Define Gull and Tern groups via class names
# ------------------------------------------------------------

gull_classes = classes[classes["class_name"].str.contains("Gull")]
tern_classes = classes[classes["class_name"].str.contains("Tern")]

gull_ids = gull_classes["class_id"].tolist()
tern_ids = tern_classes["class_id"].tolist()

print("=== Species Counts ===")
print("Gull species:", len(gull_ids))
print("Tern species:", len(tern_ids))

# ------------------------------------------------------------
# Image counts
# ------------------------------------------------------------

def count_images(class_ids):
    df = labels[labels["class_id"].isin(class_ids)]
    if USE_TRAIN_ONLY:
        df = df.merge(split, on="image_id")
        df = df[df["is_train"] == 1]
    return df.shape[0]

print("\n=== Image Counts ===")
print("USE_TRAIN_ONLY:", USE_TRAIN_ONLY)
print("Gull images:", count_images(gull_ids))
print("Tern images:", count_images(tern_ids))

# ------------------------------------------------------------
# Group-level means (class-level probabilities)
# ------------------------------------------------------------

gull_idx_class = np.array(gull_ids) - 1
tern_idx_class = np.array(tern_ids) - 1

def group_means(idx):
    return (
        white[idx].mean(),
        gull[idx].mean(),
        solid[idx].mean()
    )

mean_gull = group_means(gull_idx_class)
mean_tern = group_means(tern_idx_class)

print("\n=== Class-Level Means ===")
print("Gulls  [white, gull-like, solid]:", tuple(round(x,3) for x in mean_gull))
print("Terns  [white, gull-like, solid]:", tuple(round(x,3) for x in mean_tern))
print("Delta  (Gull - Tern):", tuple(round(a-b,3) for a,b in zip(mean_gull,mean_tern)))

# ------------------------------------------------------------
# Correlation matrix among triple (restricted to gull+tern classes)
# ------------------------------------------------------------

restricted_idx = np.concatenate([gull_idx_class, tern_idx_class])

M = np.vstack([
    white[restricted_idx],
    gull[restricted_idx],
    solid[restricted_idx]
]).T

corr_matrix = np.corrcoef(M, rowvar=False)

print("\n=== Correlation Matrix (Gulls + Terns Only) ===")
print("           white    gull_like   solid")
print(np.round(corr_matrix, 3))

# ------------------------------------------------------------
# Association with binary outcome (image-level)
# ------------------------------------------------------------

# Build image-level dataset for gull vs tern
dfA = labels[labels["class_id"].isin(gull_ids)].copy()
dfB = labels[labels["class_id"].isin(tern_ids)].copy()

dfA["y"] = 1
dfB["y"] = 0
dfY = pd.concat([dfA, dfB], ignore_index=True)

if USE_TRAIN_ONLY:
    dfY = dfY.merge(split, on="image_id")
    dfY = dfY[dfY["is_train"] == 1]

keep_ids = set(dfY["image_id"])

# Extract concept labels
white_id = white_idx + 1
gull_id  = gull_idx + 1
solid_id = solid_idx + 1

rows = []
for chunk in pd.read_csv(
    img_attr_path,
    delim_whitespace=True,
    header=None,
    names=["image_id","attr_id","is_present","certainty","time"],
    chunksize=1_000_000,
    engine="c",
    on_bad_lines="skip",
):
    chunk = chunk[chunk["image_id"].isin(keep_ids)]
    chunk = chunk[chunk["attr_id"].isin([white_id, gull_id, solid_id])]
    if not chunk.empty:
        rows.append(chunk)

df_attr = pd.concat(rows)

X = df_attr.pivot_table(
    index="image_id",
    columns="attr_id",
    values="is_present",
    aggfunc="max"
).fillna(0)

for col in [white_id,gull_id,solid_id]:
    if col not in X.columns:
        X[col] = 0

X = X[[white_id,gull_id,solid_id]]
X.columns = ["white","gull_like","solid"]

X = dfY[["image_id","y"]].merge(X, left_on="image_id", right_index=True)

print("\n=== Concept ↔ Binary Outcome Association (Point-Biserial r) ===")

for col in ["white","gull_like","solid"]:
    r,p = pointbiserialr(X["y"], X[col])
    print(f"{col:10s} r={r:.3f}, p={p:.3e}")

=== Species Counts ===
Gull species: 8
Tern species: 7

=== Image Counts ===
USE_TRAIN_ONLY: False
Gull images: 469
Tern images: 418

=== Class-Level Means ===
Gulls  [white, gull-like, solid]: (np.float64(67.233), np.float64(54.236), np.float64(45.709))
Terns  [white, gull-like, solid]: (np.float64(60.244), np.float64(42.502), np.float64(59.948))
Delta  (Gull - Tern): (np.float64(6.989), np.float64(11.734), np.float64(-14.239))

=== Correlation Matrix (Gulls + Terns Only) ===
           white    gull_like   solid
[[ 1.     0.382  0.255]
 [ 0.382  1.    -0.399]
 [ 0.255 -0.399  1.   ]]


/tmp/ipykernel_9583/416109382.py:179: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  for chunk in pd.read_csv(



=== Concept ↔ Binary Outcome Association (Point-Biserial r) ===
white      r=0.113, p=7.855e-04
gull_like  r=0.091, p=6.524e-03
solid      r=-0.098, p=3.420e-03


In [12]:
bad_line_no = 709498
with open(img_attr_path, "r") as f:
    for i, line in enumerate(f, start=1):
        if i == bad_line_no:
            print(repr(line))
            print("tokens:", line.split())
            break

'2275 10 0 1 0  1.509\n'
tokens: ['2275', '10', '0', '1', '0', '1.509']


In [15]:
"""
CUB Gull-vs-Tern Concept-Frustration Pipeline (CLIP ViT-B/16 embeddings)
======================================================================

Adapts the sarcasm pipeline to CUB images:

- Binary outcome Y: Gull (1) vs Tern (0) based on class name substring match.
- X: CLIP ViT-B/16 image embeddings (frozen).
- C: three CUB image attributes (is_present in image_attribute_labels.txt):
    1) has_upperparts_color::white
    2) has_shape::gull-like
    3) has_wing_pattern::solid

CBM1 uses first 2 concepts (white, gull_like)
CBM2 uses all 3 concepts (white, gull_like, solid)

Outputs per-fold:
- BB acc/F1
- CBM1 acc/F1 + frustration metrics
- CBM2 acc/F1 + frustration metrics + pair12 metrics
- C3 predictability from frustrated vs non-frustrated atoms (ridge)

Notes:
- Requires: torch, transformers, pandas, numpy, pillow, scipy
- Works with your nested tree:
    ./CUB_200_2011/attributes.txt
    ./CUB_200_2011/CUB_200_2011/{images.txt, classes.txt, attributes/...}

"""

import os
import time
import hashlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from scipy.stats import pointbiserialr

from transformers import CLIPProcessor, CLIPModel


# ============================================================
# Config
# ============================================================

# ---- dataset / task
USE_TRAIN_ONLY = False          # for final eval you might set True, but start with False
MAX_PER_CLASS = None            # e.g. 80 to cap per-class images; None = use all
SEED_DATA = 123

# ---- CUB root folder name in cwd
CUB_BASE_NAME = "CUB_200_2011"

# ---- Concepts (MUST match attributes.txt strings exactly)
CONCEPT_NAMES = [
    "has_upperparts_color::white",
    "has_shape::gull-like",
    "has_wing_pattern::solid",
]
# CBM1 uses first 2 concepts; CBM2 uses all 3.

# ---- Embeddings
CLIP_MODEL_NAME = "openai/clip-vit-base-patch16"
BATCH_SIZE_EMB = 64
IMAGE_SIZE = 224  # CLIP processor handles resizing; kept for clarity

# ---- Folds / training
K_FOLDS = 10
SEED_FOLDS = 777

HIDDEN_BB = 128
EPOCHS_BB = 60

EPOCHS_CBM = 30
LAMBDA_C = 1.0

K_SAE = 300
EPOCHS_SAE = 60

# ---- Fisher keep window
P_LO, P_HI = 0.2, 0.8
MIN_KEEP = 200

PAIR_SOFT_EPS = 1e-6
RIDGE_LAM_C3 = 1e-3

# ---- Normalization toggles
STANDARDIZE_X_PER_FOLD = True
NORMALIZE_C_FOR_CBM = True
NORM_EPS = 1e-6

# ---- output
OUT_DIR = "cub_gull_tern_frustration_runs"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# Task hashing -> cache-safe filenames
# ============================================================

def _task_signature() -> str:
    payload = {
        "DATASET": "CUB-200-2011",
        "TASK": "Gull_vs_Tern",
        "USE_TRAIN_ONLY": bool(USE_TRAIN_ONLY),
        "MAX_PER_CLASS": MAX_PER_CLASS,
        "SEED_DATA": int(SEED_DATA),
        "CONCEPTS": list(CONCEPT_NAMES),
        "CLIP_MODEL": CLIP_MODEL_NAME,
        "K_FOLDS": int(K_FOLDS),
    }
    b = repr(payload).encode("utf-8")
    return hashlib.md5(b).hexdigest()[:10]

TASK_SIG = _task_signature()

CACHE_NPZ = os.path.join(OUT_DIR, f"cache_cub_gulltern_{TASK_SIG}.npz")
RESULTS_CSV = os.path.join(OUT_DIR, f"results_cub_gulltern_{TASK_SIG}_K{K_FOLDS}_t{int(time.time())}.csv")


# ============================================================
# 1) Models (unchanged)
# ============================================================

class BlackBoxMLP(nn.Module):
    def __init__(self, r: int, hidden: int = 128):
        super().__init__()
        self.fc1 = nn.Linear(r, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, x, return_hidden: bool = False):
        z = self.fc1(x)
        h = self.act(z)
        logit = self.fc2(h).squeeze(-1)
        if return_hidden:
            return logit, z, h
        return logit


class LinearCBM(nn.Module):
    def __init__(self, xdim: int, k_known: int):
        super().__init__()
        self.concept = nn.Linear(xdim, k_known)
        self.task = nn.Linear(k_known, 1)

    def forward(self, x):
        c = self.concept(x)
        y = self.task(c).squeeze(-1)
        return c, y


class SparseAE(nn.Module):
    def __init__(self, xdim: int, K: int):
        super().__init__()
        self.W = nn.Linear(xdim, K, bias=True)
        self.D = nn.Parameter(torch.randn(K, xdim) * 0.02)

    def forward(self, x):
        s = torch.relu(self.W(x))
        x_hat = s @ self.D
        return s, x_hat


# ============================================================
# 2) Helpers
# ============================================================

@torch.no_grad()
def accuracy_from_logits(logits: torch.Tensor, y_true_np: np.ndarray) -> float:
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= 0.5).astype(np.int64)
    return float((preds == y_true_np).mean())

@torch.no_grad()
def f1_from_logits(logits: torch.Tensor, y_true_np: np.ndarray, thresh: float = 0.5) -> float:
    y_true = np.asarray(y_true_np, dtype=np.int64).reshape(-1)
    probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
    y_pred = (probs >= thresh).astype(np.int64)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    denom = (2 * tp + fp + fn)
    return float((2 * tp) / denom) if denom > 0 else 0.0


# ============================================================
# 3) Fisher helpers (unchanged)
# ============================================================

@torch.no_grad()
def bb_predict_proba(bb: BlackBoxMLP, X: np.ndarray, *, device="cpu") -> np.ndarray:
    bb.eval()
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    logits = bb(Xt)
    return torch.sigmoid(logits).detach().cpu().numpy()


@torch.no_grad()
def compute_fisher_on_input_x(bb: BlackBoxMLP, X: np.ndarray, *, device="cpu", ridge=1e-6):
    bb.eval()
    Xt = torch.tensor(X, dtype=torch.float32, device=device)

    logits, z, _ = bb(Xt, return_hidden=True)
    p = torch.sigmoid(logits)
    s = p * (1 - p)
    m = (z > 0).float()

    W_H = bb.fc1.weight
    w_l = bb.fc2.weight.squeeze(0)

    U = m * w_l.unsqueeze(0)
    G = U @ W_H

    Fm = (G.T @ (G * s.unsqueeze(1))) / float(X.shape[0])
    Fm = Fm + ridge * torch.eye(Fm.shape[0], device=device)
    return Fm.cpu().numpy()


# ============================================================
# 4) Training (unchanged)
# ============================================================

def train_bb_minibatch(X_tr, y_tr, X_te, y_te, *, hidden=128, epochs=30, batch_size=512, lr=1e-3, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    r = X_tr.shape[1]
    model = BlackBoxMLP(r, hidden=hidden).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)
    Xv = torch.tensor(X_te, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb, yb = Xt[b], yt[b]
            opt.zero_grad()
            loss = bce(model(xb), yb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        acc_te = accuracy_from_logits(model(Xv), y_te)
    return model, acc_te


def train_cbm_linear_minibatch(X_tr, Ck_tr, y_tr, X_te, Ck_te, y_te, *,
                              epochs=30, batch_size=512, lr=1e-3, lambda_c=1.0, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    xdim = X_tr.shape[1]
    k = Ck_tr.shape[1]
    model = LinearCBM(xdim, k).to(device)

    opt = optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    Ct = torch.tensor(Ck_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)

    Xv = torch.tensor(X_te, dtype=torch.float32, device=device)
    Cv = torch.tensor(Ck_te, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb, cb, yb = Xt[b], Ct[b], yt[b]
            opt.zero_grad()
            c_hat, logit = model(xb)
            loss = bce(logit, yb) + lambda_c * mse(c_hat, cb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        c_hat_te, logit_te = model(Xv)
        acc_te = accuracy_from_logits(logit_te, y_te)
        mse_te = float(((c_hat_te - Cv)**2).mean().cpu().item())

    return model, acc_te, mse_te


def train_sae_minibatch(X_tr, *, K=60, epochs=60, batch_size=512, lr=2e-3, l1=1e-3, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    xdim = X_tr.shape[1]
    model = SparseAE(xdim, K).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb = Xt[b]
            opt.zero_grad()
            s_code, xhat = model(xb)
            loss = ((xhat - xb)**2).mean() + l1 * s_code.abs().mean()
            loss.backward()
            opt.step()

    return model


# ============================================================
# 5) Geometry cosines (unchanged)
# ============================================================

def fisher_cosine_matrix(W: np.ndarray, D: np.ndarray, Fm: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Fsym = 0.5 * (Fm + Fm.T)
    WFW = np.einsum("ih,hk,ik->i", W, Fsym, W)
    DFD = np.einsum("jh,hk,jk->j", D, Fsym, D)
    Wn = np.sqrt(np.maximum(WFW, eps))
    Dn = np.sqrt(np.maximum(DFD, eps))
    num = W @ Fsym @ D.T
    return num / (Wn[:, None] * Dn[None, :] + eps)


def euclid_cosine_matrix(W: np.ndarray, D: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Wn = np.sqrt(np.sum(W * W, axis=1, keepdims=True) + eps)
    Dn = np.sqrt(np.sum(D * D, axis=1, keepdims=True) + eps)
    return (W @ D.T) / (Wn @ Dn.T)


def fisher_cosine_self(W: np.ndarray, Fm: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Fsym = 0.5 * (Fm + Fm.T)
    WFW = np.einsum("ih,hk,ik->i", W, Fsym, W)
    Wn = np.sqrt(np.maximum(WFW, eps))
    num = W @ Fsym @ W.T
    return num / (Wn[:, None] * Wn[None, :] + eps)


def euclid_cosine_self(W: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Wn = np.sqrt(np.sum(W * W, axis=1) + eps)
    num = W @ W.T
    return num / (Wn[:, None] * Wn[None, :] + eps)


# ============================================================
# 6) Pair frustration + faithfulness helpers (unchanged)
# ============================================================

def pair_soft_frustration_metric(S: np.ndarray, Z: np.ndarray, *, eps: float = 1e-6) -> dict:
    k, K = S.shape
    if k < 2:
        return {
            "pair_soft_mean": 0.0, "pair_soft_max": 0.0, "pair_soft_nonzero_frac": 0.0,
            "pair_soft_pairs": 0, "pair_soft_eps": float(eps), "mean_abs_Z": 0.0,
            "pair_raw_mean": 0.0, "pair_raw_max": 0.0, "pair_raw_nonzero_frac": 0.0,
        }

    soft_scores, raw_scores = [], []
    soft_nonzero = 0
    raw_nonzero = 0
    total_pairs = 0

    for l in range(k):
        for r in range(l + 1, k):
            total_pairs += 1
            z = float(Z[l, r])
            if z == 0.0:
                soft_scores.append(0.0); raw_scores.append(0.0)
                continue

            prod = S[l, :] * S[r, :]
            zsign = np.sign(z)
            psign = np.sign(prod)

            contr_mask = (psign != 0) & (psign != zsign)
            if not np.any(contr_mask):
                soft_scores.append(0.0); raw_scores.append(0.0)
                continue

            best_raw = float(np.max(np.abs(prod[contr_mask])))
            raw_scores.append(best_raw)
            if best_raw > 0.0:
                raw_nonzero += 1

            best_soft = best_raw / (abs(z) + eps)
            soft_scores.append(best_soft)
            if best_soft > 0.0:
                soft_nonzero += 1

    soft_scores = np.asarray(soft_scores, dtype=float)
    raw_scores = np.asarray(raw_scores, dtype=float)

    tri = np.triu_indices(k, 1)
    mean_abs_Z = float(np.mean(np.abs(Z[tri])))

    return {
        "pair_soft_mean": float(soft_scores.mean()) if soft_scores.size else 0.0,
        "pair_soft_max": float(soft_scores.max()) if soft_scores.size else 0.0,
        "pair_soft_nonzero_frac": float(soft_nonzero / total_pairs) if total_pairs else 0.0,
        "pair_soft_pairs": int(total_pairs),
        "pair_soft_eps": float(eps),
        "mean_abs_Z": float(mean_abs_Z),
        "pair_raw_mean": float(raw_scores.mean()) if raw_scores.size else 0.0,
        "pair_raw_max": float(raw_scores.max()) if raw_scores.size else 0.0,
        "pair_raw_nonzero_frac": float(raw_nonzero / total_pairs) if total_pairs else 0.0,
    }


def frob_norm(A: np.ndarray) -> float:
    return float(np.sqrt(np.sum(A * A)))


def frob_abs_rel(A: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> dict:
    diff = A - B
    abs_f = frob_norm(diff)
    denom = frob_norm(A) + eps
    rel_f = abs_f / denom
    return {"frob_abs": float(abs_f), "frob_rel": float(rel_f)}


def cov_matrix(X: np.ndarray) -> np.ndarray:
    return np.cov(X, rowvar=False, bias=False)


def corr_from_cov(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    d = np.sqrt(np.maximum(np.diag(C), eps))
    return C / (d[:, None] * d[None, :] + eps)


# ============================================================
# 6.5) C3 prediction from frustrated vs non-frustrated atoms (unchanged)
# ============================================================

def _select_frustrated_atoms_pair12(S: np.ndarray, Z: np.ndarray, *, tiny: float = 1e-15) -> np.ndarray:
    z = float(Z[0, 1])
    if z == 0.0:
        return np.array([], dtype=np.int64)

    prod = S[0, :] * S[1, :]
    mask_nz = np.abs(prod) > tiny
    zsign = np.sign(z)
    psign = np.sign(prod)

    contr = mask_nz & (psign != 0) & (psign != zsign)
    return np.where(contr)[0].astype(np.int64)


def _project_X_onto_atoms(X: np.ndarray, D: np.ndarray, idx: np.ndarray) -> np.ndarray:
    if idx.size == 0:
        return np.zeros((X.shape[0], 0), dtype=np.float32)
    return (X @ D[idx, :].T).astype(np.float32)


def _ridge_predict(Xtr: np.ndarray, ytr: np.ndarray, Xte: np.ndarray, *, lam: float = 1e-3):
    Xtr = np.asarray(Xtr, dtype=np.float64)
    Xte = np.asarray(Xte, dtype=np.float64)
    ytr = np.asarray(ytr, dtype=np.float64).reshape(-1)

    x_mu = Xtr.mean(axis=0, keepdims=True) if Xtr.shape[1] else np.zeros((1, 0), dtype=np.float64)
    y_mu = float(ytr.mean())

    Xc = Xtr - x_mu
    yc = ytr - y_mu

    d = Xc.shape[1]
    if d == 0:
        yhat = np.full((Xte.shape[0],), y_mu, dtype=np.float32)
        return yhat, (y_mu, np.zeros((0,), dtype=np.float32))

    A = Xc.T @ Xc + lam * np.eye(d)
    b = np.linalg.solve(A, Xc.T @ yc)
    b0 = y_mu - float((x_mu @ b.reshape(-1, 1)).squeeze())
    yhat = (Xte @ b) + b0
    return yhat.astype(np.float32), (float(b0), b.astype(np.float32))


def _mse_r2(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    mse = float(np.mean((y_true - y_pred) ** 2))
    var = float(np.var(y_true))
    r2 = float(1.0 - mse / (var + 1e-12))
    return mse, r2


# ============================================================
# 7) Data build: CLIP embeddings X + CUB attributes C + Y
# ============================================================

def _autodetect_cub_paths():
    """
    Supports:
      (A) standard: ./CUB_200_2011/{images.txt, attributes/attributes.txt, ...}
      (B) nested:   ./CUB_200_2011/attributes.txt + ./CUB_200_2011/CUB_200_2011/{...}
    """
    base = os.path.join(os.getcwd(), CUB_BASE_NAME)

    std_root = base
    std_attr_names = os.path.join(std_root, "attributes", "attributes.txt")

    nested_root = os.path.join(base, "CUB_200_2011")
    nested_attr_names = os.path.join(base, "attributes.txt")

    def _exists_all(paths):
        return all(os.path.exists(p) for p in paths)

    if _exists_all([os.path.join(std_root, "images.txt"), std_attr_names]):
        return std_root, std_attr_names, os.path.join(std_root, "images")
    if _exists_all([os.path.join(nested_root, "images.txt"), nested_attr_names]):
        return nested_root, nested_attr_names, os.path.join(nested_root, "images")

    raise FileNotFoundError(
        "Could not auto-detect CUB layout. Expected ./CUB_200_2011/... per your setup."
    )


def _read_two_col(path, c1, c2):
    return pd.read_csv(path, sep=r"\s+", header=None, names=[c1, c2], engine="python")


def _get_attr_id(attr_names_df: pd.DataFrame, attr_name: str) -> int:
    row = attr_names_df[attr_names_df["attr_name"] == attr_name]
    if row.empty:
        # show partials to help debugging
        partial = attr_names_df[attr_names_df["attr_name"].str.contains(attr_name.split("::")[-1], regex=False)]
        raise ValueError(f"Attribute not found: {attr_name}\nPartial matches:\n{partial.head(25)}")
    return int(row.iloc[0]["attr_id"])  # 1-based


@torch.no_grad()
def _embed_images_clip(image_paths: list[str], model_name: str, *, batch_size: int, device):
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device)
    model.eval()

    feats = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_imgs = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            batch_imgs.append(img)

        inputs = processor(images=batch_imgs, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        f = model.get_image_features(**inputs)  # [B, d]
        f = f / (f.norm(dim=-1, keepdim=True) + 1e-12)  # normalize
        feats.append(f.detach().cpu())

    X = torch.cat(feats, dim=0).numpy().astype(np.float32)
    return X


def build_or_load_cached_dataset():
    if os.path.exists(CACHE_NPZ):
        print("Loading cached dataset:", CACHE_NPZ)
        z = np.load(CACHE_NPZ, allow_pickle=True)
        return (z["X"].astype(np.float32),
                z["C"].astype(np.float32),
                z["Y"].astype(np.int64),
                z["image_ids"].astype(np.int64),
                z["rel_paths"].tolist())

    CUB_ROOT, ATTR_NAMES_PATH, IMAGES_DIR = _autodetect_cub_paths()
    print("✅ Using CUB_ROOT:", CUB_ROOT)
    print("✅ Using ATTR_NAMES_PATH:", ATTR_NAMES_PATH)
    print("✅ Using IMAGES_DIR:", IMAGES_DIR)

    images = _read_two_col(os.path.join(CUB_ROOT, "images.txt"), "image_id", "rel_path")
    labels = _read_two_col(os.path.join(CUB_ROOT, "image_class_labels.txt"), "image_id", "class_id")
    split  = _read_two_col(os.path.join(CUB_ROOT, "train_test_split.txt"), "image_id", "is_train")
    classes = _read_two_col(os.path.join(CUB_ROOT, "classes.txt"), "class_id", "class_name")
    attr_names = _read_two_col(ATTR_NAMES_PATH, "attr_id", "attr_name")

    # group definition by class name
    gull_classes = classes[classes["class_name"].str.contains("Gull", regex=False)]
    tern_classes = classes[classes["class_name"].str.contains("Tern", regex=False)]
    gull_ids = gull_classes["class_id"].tolist()
    tern_ids = tern_classes["class_id"].tolist()

    print(f"[DEBUG] Gull species={len(gull_ids)} | Tern species={len(tern_ids)}")

    # image-level rows for just gull+tern
    dfA = labels[labels["class_id"].isin(gull_ids)].copy()
    dfB = labels[labels["class_id"].isin(tern_ids)].copy()
    dfA["Y"] = 1
    dfB["Y"] = 0
    dfY = pd.concat([dfA, dfB], ignore_index=True)

    if USE_TRAIN_ONLY:
        dfY = dfY.merge(split, on="image_id", how="inner")
        dfY = dfY[dfY["is_train"] == 1].copy()

    # optional cap per class (keeps balance)
    if MAX_PER_CLASS is not None:
        rng = np.random.default_rng(SEED_DATA)
        kept = []
        for cid, sub in dfY.groupby("class_id", sort=False):
            if len(sub) <= MAX_PER_CLASS:
                kept.append(sub)
            else:
                take = rng.choice(sub.index.to_numpy(), size=MAX_PER_CLASS, replace=False)
                kept.append(sub.loc[take])
        dfY = pd.concat(kept, ignore_index=True)

    dfY = dfY.merge(images, on="image_id", how="left")
    dfY = dfY.dropna(subset=["rel_path"]).reset_index(drop=True)

    # absolute paths
    abs_paths = [os.path.join(IMAGES_DIR, rp) for rp in dfY["rel_path"].tolist()]
    missing = [p for p in abs_paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Missing {len(missing)} image files. Example: {missing[0]}")

    Y = dfY["Y"].to_numpy(dtype=np.int64)
    image_ids = dfY["image_id"].to_numpy(dtype=np.int64)
    rel_paths = dfY["rel_path"].tolist()

    print(f"[DEBUG] N={len(Y)} | pos_rate(gull)={Y.mean():.3f} | gull_images={int(Y.sum())} tern_images={int((1-Y).sum())}")

    # Map concept names -> attribute IDs (1-based in file)
    attr_ids = [_get_attr_id(attr_names, nm) for nm in CONCEPT_NAMES]
    print("[DEBUG] Concept attr_ids (1-based):", list(zip(CONCEPT_NAMES, attr_ids)))

    # Load per-image attribute labels for these 3 attr_ids
    img_attr_path = os.path.join(CUB_ROOT, "attributes", "image_attribute_labels.txt")
    keep_ids = set(image_ids.tolist())
    wanted_attr_ids = set(attr_ids)

    rows = []
    for chunk in pd.read_csv(
        img_attr_path,
        sep=r"\s+",
        header=None,
        names=["image_id", "attr_id", "is_present", "certainty", "time"],
        chunksize=1_000_000,
        engine="c",
        on_bad_lines="skip",
    ):
        chunk = chunk[chunk["image_id"].isin(keep_ids)]
        chunk = chunk[chunk["attr_id"].isin(wanted_attr_ids)]
        if not chunk.empty:
            rows.append(chunk)

    if not rows:
        raise RuntimeError("No attribute rows found for selected images/attributes (check paths).")

    df_attr = pd.concat(rows, ignore_index=True)

    C_wide = df_attr.pivot_table(
        index="image_id",
        columns="attr_id",
        values="is_present",
        aggfunc="max"
    ).fillna(0.0)

    # ensure all cols exist and in the right order
    for aid in attr_ids:
        if aid not in C_wide.columns:
            C_wide[aid] = 0.0
    C_wide = C_wide[attr_ids]
    C_wide = C_wide.reindex(image_ids).fillna(0.0)

    C = C_wide.to_numpy(dtype=np.float32)

    # Quick sanity stats (association)
    print("\n[DEBUG] Concept ↔ Y (point-biserial):")
    for j, nm in enumerate(CONCEPT_NAMES):
        r, p = pointbiserialr(Y.astype(float), C[:, j].astype(float))
        print(f"  {nm:35s} r={r:+.3f} p={p:.2e}")

    # Embed images with CLIP
    print("\nComputing CLIP embeddings...")
    X = _embed_images_clip(abs_paths, CLIP_MODEL_NAME, batch_size=BATCH_SIZE_EMB, device=device)
    print("X shape:", X.shape, "C shape:", C.shape, "Y shape:", Y.shape)

    np.savez_compressed(
        CACHE_NPZ,
        X=X, C=C, Y=Y,
        image_ids=image_ids,
        rel_paths=np.array(rel_paths, dtype=object),
    )
    print("Saved cache:", CACHE_NPZ)
    return X, C, Y, image_ids, rel_paths


# ============================================================
# 8) Stratified folds (unchanged)
# ============================================================

def make_stratified_folds(y: np.ndarray, k_folds: int, seed: int):
    y = np.asarray(y, dtype=np.int64).reshape(-1)
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    rng = np.random.default_rng(seed)
    rng.shuffle(pos); rng.shuffle(neg)

    pos_folds = np.array_split(pos, k_folds)
    neg_folds = np.array_split(neg, k_folds)

    folds = []
    all_idx = np.arange(len(y), dtype=np.int64)
    for k in range(k_folds):
        te = np.concatenate([pos_folds[k], neg_folds[k]])
        te = np.unique(te)
        mask = np.ones(len(y), dtype=bool)
        mask[te] = False
        tr = all_idx[mask]
        folds.append((tr, te))
    return folds


# ============================================================
# 9) One fold run (same structure, now C is 3 concepts)
# ============================================================

def run_one_fold(
    *,
    fold_id: int,
    seed: int,
    X: np.ndarray,
    C: np.ndarray,
    y: np.ndarray,
    tr_idx: np.ndarray,
    te_idx: np.ndarray,
    device="cpu",
    p_lo=0.2,
    p_hi=0.8,
    min_keep=200,
    K_sae=60,
    pair_soft_eps=1e-6,
    ridge_lam_c3=1e-3,
):
    r = X.shape[1]

    # covariance on raw C (full 3 concepts)
    B = np.cov(C, rowvar=False, bias=False).astype(np.float32)

    X_tr, X_te = X[tr_idx].astype(np.float32), X[te_idx].astype(np.float32)
    y_tr, y_te = y[tr_idx], y[te_idx]

    if STANDARDIZE_X_PER_FOLD:
        x_mu = X_tr.mean(axis=0, keepdims=True)
        x_sd = X_tr.std(axis=0, keepdims=True) + NORM_EPS
        X_tr = (X_tr - x_mu) / x_sd
        X_te = (X_te - x_mu) / x_sd

    bb, bb_acc = train_bb_minibatch(
        X_tr, y_tr, X_te, y_te,
        hidden=HIDDEN_BB, epochs=EPOCHS_BB, batch_size=512, lr=1e-3,
        seed=seed, device=device
    )

    with torch.no_grad():
        bb_logits_te = bb(torch.tensor(X_te, dtype=torch.float32, device=device))
        bb_f1 = f1_from_logits(bb_logits_te, y_te, thresh=0.5)

    p_lo_use, p_hi_use = (p_lo, p_hi) if p_lo < p_hi else (p_hi, p_lo)
    p_tr = bb_predict_proba(bb, X_tr, device=device)
    keep = np.where((p_tr >= p_lo_use) & (p_tr <= p_hi_use))[0]
    if keep.size < min_keep:
        order = np.argsort(np.abs(p_tr - 0.5))
        keep = order[:min_keep]
    Fm = compute_fisher_on_input_x(bb, X_tr[keep], device=device, ridge=1e-6)

    sae = train_sae_minibatch(
        X_tr,
        K=K_sae, epochs=EPOCHS_SAE, batch_size=512, lr=2e-3, l1=1e-3,
        seed=seed, device=device
    )
    D = sae.D.detach().cpu().numpy().astype(np.float32)

    C_tr_raw = C[tr_idx].astype(np.float32)
    C_te_raw = C[te_idx].astype(np.float32)

    if NORMALIZE_C_FOR_CBM:
        c_mu = C_tr_raw.mean(axis=0, keepdims=True)
        c_sd = C_tr_raw.std(axis=0, keepdims=True) + NORM_EPS
        C_tr_norm = (C_tr_raw - c_mu) / c_sd
        C_te_norm = (C_te_raw - c_mu) / c_sd
    else:
        C_tr_norm, C_te_norm = C_tr_raw, C_te_raw

    def _run_cbm_block(k_known: int, tag: str):
        Ck_tr = C_tr_norm[:, :k_known].astype(np.float32)
        Ck_te = C_te_norm[:, :k_known].astype(np.float32)
        B_known = np.array(B[:k_known, :k_known], dtype=float)

        cbm, cbm_acc, cbm_mse = train_cbm_linear_minibatch(
            X_tr, Ck_tr, y_tr,
            X_te, Ck_te, y_te,
            epochs=EPOCHS_CBM, batch_size=512, lr=1e-3,
            lambda_c=LAMBDA_C, seed=seed, device=device
        )
        Wc = cbm.concept.weight.detach().cpu().numpy().astype(np.float32)

        with torch.no_grad():
            Xv = torch.tensor(X_te, dtype=torch.float32, device=device)
            c_hat_te, logit_te = cbm(Xv)
            C_hat_te_norm = c_hat_te.detach().cpu().numpy().astype(np.float32)
            cbm_f1 = f1_from_logits(logit_te, y_te, thresh=0.5)

        if NORMALIZE_C_FOR_CBM:
            mu_k = c_mu[:, :k_known]
            sd_k = c_sd[:, :k_known]
            C_hat_te_raw = (C_hat_te_norm * sd_k) + mu_k
        else:
            C_hat_te_raw = C_hat_te_norm

        S_f = fisher_cosine_matrix(Wc, D, Fm)
        S_e = euclid_cosine_matrix(Wc, D)
        Z_f = fisher_cosine_self(Wc, Fm)
        Z_e = euclid_cosine_self(Wc)

        pairF = pair_soft_frustration_metric(S_f, Z_f, eps=PAIR_SOFT_EPS)
        pairE = pair_soft_frustration_metric(S_e, Z_e, eps=PAIR_SOFT_EPS)

        Sd = frob_abs_rel(S_f, S_e)

        Cov_hat = cov_matrix(C_hat_te_raw)
        Corr_hat = corr_from_cov(Cov_hat)
        Corr_B = corr_from_cov(B_known)
        cov_diff = frob_abs_rel(Cov_hat, B_known)
        corr_diff = frob_abs_rel(Corr_hat, Corr_B)

        out = {
            f"{tag}_k_known": int(k_known),
            f"{tag}_acc": float(cbm_acc),
            f"{tag}_f1": float(cbm_f1),
            f"{tag}_mse": float(cbm_mse),

            f"{tag}_F_pair_raw_mean": float(pairF["pair_raw_mean"]),
            f"{tag}_E_pair_raw_mean": float(pairE["pair_raw_mean"]),
            f"{tag}_F_pair_raw_max": float(pairF["pair_raw_max"]),
            f"{tag}_E_pair_raw_max": float(pairE["pair_raw_max"]),

            f"{tag}_S_frob_abs": float(Sd["frob_abs"]),
            f"{tag}_S_frob_rel": float(Sd["frob_rel"]),

            f"{tag}_Cov_frob_abs": float(cov_diff["frob_abs"]),
            f"{tag}_Cov_frob_rel": float(cov_diff["frob_rel"]),
            f"{tag}_Corr_frob_abs": float(corr_diff["frob_abs"]),
            f"{tag}_Corr_frob_rel": float(corr_diff["frob_rel"]),
        }
        return out, (S_f, Z_f, Wc)

    # CBM1 (C1,C2)
    cbm1_metrics, (S1_f, Z1_f, _W1) = _run_cbm_block(2, "cbm1")

    # C3 predictability from atoms defined by CBM1 pair (C1,C2)
    C3_tr = C_tr_raw[:, 2].astype(np.float32)
    C3_te = C_te_raw[:, 2].astype(np.float32)

    frust_idx_full = _select_frustrated_atoms_pair12(S1_f, Z1_f, tiny=1e-15)
    n_frust_full = int(frust_idx_full.size)

    all_idx_atoms = np.arange(D.shape[0], dtype=np.int64)
    non_frust_pool = np.setdiff1d(all_idx_atoms, frust_idx_full, assume_unique=False)
    n_nonfrust_pool = int(non_frust_pool.size)

    rng_atoms = np.random.default_rng(seed + 99991 + fold_id)
    m_match = int(min(n_frust_full, n_nonfrust_pool))

    if m_match > 0:
        frust_idx = rng_atoms.choice(frust_idx_full, size=m_match, replace=False)
        non_frust_idx = rng_atoms.choice(non_frust_pool, size=m_match, replace=False)
    else:
        frust_idx = np.array([], dtype=np.int64)
        non_frust_idx = np.array([], dtype=np.int64)

    PhiF_tr = _project_X_onto_atoms(X_tr, D, frust_idx)
    PhiF_te = _project_X_onto_atoms(X_te, D, frust_idx)
    PhiN_tr = _project_X_onto_atoms(X_tr, D, non_frust_idx)
    PhiN_te = _project_X_onto_atoms(X_te, D, non_frust_idx)

    yhatF_te, _ = _ridge_predict(PhiF_tr, C3_tr, PhiF_te, lam=ridge_lam_c3)
    yhatN_te, _ = _ridge_predict(PhiN_tr, C3_tr, PhiN_te, lam=ridge_lam_c3)

    C3_mse_frust, C3_r2_frust = _mse_r2(C3_te, yhatF_te)
    C3_mse_nonfrust, C3_r2_nonfrust = _mse_r2(C3_te, yhatN_te)

    ridge_metrics = {
        "n_frust_atoms_full": int(n_frust_full),
        "n_nonfrust_pool": int(n_nonfrust_pool),
        "n_atoms_matched": int(m_match),
        "C3_ridge_lam": float(ridge_lam_c3),
        "C3_mse_frust_atoms": float(C3_mse_frust),
        "C3_r2_frust_atoms": float(C3_r2_frust),
        "C3_mse_nonfrust_atoms": float(C3_mse_nonfrust),
        "C3_r2_nonfrust_atoms": float(C3_r2_nonfrust),
    }

    # CBM2 (C1,C2,C3)
    cbm2_metrics, (S2_f, Z2_f, W2) = _run_cbm_block(3, "cbm2")

    # CBM2 pair12 metrics (restrict W2 to first 2 concepts)
    W2_12 = W2[:2, :]
    S2_f_12 = fisher_cosine_matrix(W2_12, D, Fm)
    S2_e_12 = euclid_cosine_matrix(W2_12, D)
    Z2_f_12 = fisher_cosine_self(W2_12, Fm)
    Z2_e_12 = euclid_cosine_self(W2_12)

    pairF2_12 = pair_soft_frustration_metric(S2_f_12, Z2_f_12, eps=PAIR_SOFT_EPS)
    pairE2_12 = pair_soft_frustration_metric(S2_e_12, Z2_e_12, eps=PAIR_SOFT_EPS)

    cbm2_pair12_metrics = {
        "cbm2_F_pair12_raw_mean": float(pairF2_12["pair_raw_mean"]),
        "cbm2_E_pair12_raw_mean": float(pairE2_12["pair_raw_mean"]),
        "cbm2_F_pair12_raw_max":  float(pairF2_12["pair_raw_max"]),
        "cbm2_E_pair12_raw_max":  float(pairE2_12["pair_raw_max"]),
        "cbm2_F_pair12_nonzero_frac": float(pairF2_12["pair_raw_nonzero_frac"]),
        "cbm2_E_pair12_nonzero_frac": float(pairE2_12["pair_raw_nonzero_frac"]),
    }

    row = {
        "task": "CUB_Gull_vs_Tern",
        "task_sig": str(TASK_SIG),
        "clip_model": str(CLIP_MODEL_NAME),
        "concepts": "|".join(CONCEPT_NAMES),
        "stdX_per_fold": int(STANDARDIZE_X_PER_FOLD),
        "normC_for_cbm": int(NORMALIZE_C_FOR_CBM),

        "fold": int(fold_id),
        "seed": int(seed),
        "N_total": int(len(X)),
        "N_train": int(len(tr_idx)),
        "N_test": int(len(te_idx)),
        "r": int(r),
        "p_lo": float(p_lo_use),
        "p_hi": float(p_hi_use),
        "F_keep_n": int(len(keep)),
        "bb_acc": float(bb_acc),
        "bb_f1": float(bb_f1),
        "Y_pos_rate_total": float(np.mean(y)),
        "Y_pos_rate_train": float(np.mean(y_tr)),
        "Y_pos_rate_test": float(np.mean(y_te)),
    }
    row.update(cbm1_metrics)
    row.update(ridge_metrics)
    row.update(cbm2_metrics)
    row.update(cbm2_pair12_metrics)
    return row


# ============================================================
# 10) Main
# ============================================================

def main():
    print("\n=== Running CUB Gull-vs-Tern | CLIP ViT-B/16 ===")
    print(f"    USE_TRAIN_ONLY={USE_TRAIN_ONLY} | MAX_PER_CLASS={MAX_PER_CLASS}")
    print(f"    STANDARDIZE_X_PER_FOLD={STANDARDIZE_X_PER_FOLD} | NORMALIZE_C_FOR_CBM={NORMALIZE_C_FOR_CBM}")

    X, C, Y, image_ids, rel_paths = build_or_load_cached_dataset()
    assert X.shape[0] == C.shape[0] == Y.shape[0]
    assert C.shape[1] == 3, f"Expected 3 concepts, got {C.shape[1]}"
    print("Shapes: X", X.shape, "C", C.shape, "Y", Y.shape)

    folds = make_stratified_folds(Y, K_FOLDS, SEED_FOLDS)

    rows = []
    for k, (tr_idx, te_idx) in enumerate(folds):
        print(f"\n=== Fold {k+1}/{K_FOLDS} ===")
        out = run_one_fold(
            fold_id=k,
            seed=SEED_FOLDS + k,
            X=X, C=C, y=Y,
            tr_idx=tr_idx, te_idx=te_idx,
            device=str(device),
            p_lo=P_LO, p_hi=P_HI, min_keep=MIN_KEEP,
            K_sae=K_SAE, pair_soft_eps=PAIR_SOFT_EPS,
            ridge_lam_c3=RIDGE_LAM_C3,
        )
        rows.append(out)

        print(
            f"bb_acc={out['bb_acc']:.3f} bb_f1={out['bb_f1']:.3f} | "
            f"cbm1_acc={out['cbm1_acc']:.3f} cbm1_f1={out['cbm1_f1']:.3f} "
            f"GammaF1={out['cbm1_F_pair_raw_mean']:.4f} | "
            f"cbm2_acc={out['cbm2_acc']:.3f} cbm2_f1={out['cbm2_f1']:.3f} "
            f"GammaF2(allpairs)={out['cbm2_F_pair_raw_mean']:.4f} "
            f"GammaF2(pair12)={out['cbm2_F_pair12_raw_mean']:.4f} | "
            f"nF(full/match)={out['n_frust_atoms_full']}/{out['n_atoms_matched']} "
            f"C3R2(F/N)={out['C3_r2_frust_atoms']:.3f}/{out['C3_r2_nonfrust_atoms']:.3f}"
        )

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    print("\nSaved results:", RESULTS_CSV)
    print(df.describe(include="all").T[["mean","std","min","max"]])


if __name__ == "__main__":
    main()

Using device: cuda

=== Running CUB Gull-vs-Tern | CLIP ViT-B/16 ===
    USE_TRAIN_ONLY=False | MAX_PER_CLASS=None
    STANDARDIZE_X_PER_FOLD=True | NORMALIZE_C_FOR_CBM=True
Loading cached dataset: cub_gull_tern_frustration_runs/cache_cub_gulltern_30dac27907.npz
Shapes: X (887, 512) C (887, 3) Y (887,)

=== Fold 1/10 ===
bb_acc=1.000 bb_f1=1.000 | cbm1_acc=0.517 cbm1_f1=0.295 GammaF1=0.1289 | cbm2_acc=0.539 cbm2_f1=0.696 GammaF2(allpairs)=0.0081 GammaF2(pair12)=0.0068 | nF(full/match)=24/24 C3R2(F/N)=0.075/-0.012

=== Fold 2/10 ===
bb_acc=0.989 bb_f1=0.989 | cbm1_acc=0.685 cbm1_f1=0.731 GammaF1=0.0825 | cbm2_acc=0.685 cbm2_f1=0.600 GammaF2(allpairs)=0.0047 GammaF2(pair12)=0.0011 | nF(full/match)=6/6 C3R2(F/N)=-0.016/-0.032

=== Fold 3/10 ===
bb_acc=1.000 bb_f1=1.000 | cbm1_acc=0.674 cbm1_f1=0.739 GammaF1=0.0040 | cbm2_acc=0.910 cbm2_f1=0.917 GammaF2(allpairs)=0.0000 GammaF2(pair12)=0.0000 | nF(full/match)=1/1 C3R2(F/N)=0.008/0.010

=== Fold 4/10 ===
bb_acc=0.978 bb_f1=0.978 | cbm1_acc=

In [16]:
# Paired Wilcoxon tests: CBM1 vs CBM2
# - cbm1_acc vs cbm2_acc
# - "pair12" frustration means (F and E) for CBM1 vs CBM2
#
# Assumes your pipeline saved a CSV with one row per fold.

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

RESULTS_CSV = "cub_gull_tern_frustration_runs/results_cub_gulltern_30dac27907_K10_t1772556151.csv"  # <-- change me

df = pd.read_csv(RESULTS_CSV)

def _get_col(df, preferred, fallback=None):
    if preferred in df.columns:
        return preferred
    if fallback is not None and fallback in df.columns:
        return fallback
    raise KeyError(
        f"Missing column '{preferred}'"
        + (f" (and fallback '{fallback}')" if fallback else "")
        + f". Available cols (sample): {list(df.columns)[:50]}"
    )

# Accuracy columns
col_acc_1 = _get_col(df, "cbm1_acc")
col_acc_2 = _get_col(df, "cbm2_acc")

# Pair12 metrics:
# CBM1 has only one pair when k_known=2, so its "pair mean" is effectively pair12.
# CBM2 has explicit pair12 columns in the script.
col_F_1 = _get_col(df, "cbm1_F_pair12_raw_mean", fallback="cbm1_F_pair_raw_mean")
col_F_2 = _get_col(df, "cbm2_F_pair12_raw_mean")

col_E_1 = _get_col(df, "cbm1_E_pair12_raw_mean", fallback="cbm1_E_pair_raw_mean")
col_E_2 = _get_col(df, "cbm2_E_pair12_raw_mean")

def paired_wilcoxon(a, b, name, *, zero_method="wilcox", alternative="two-sided"):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    mask = np.isfinite(a) & np.isfinite(b)
    a = a[mask]; b = b[mask]
    d = a - b

    # scipy wilcoxon can error if all diffs are zero
    if np.allclose(d, 0):
        print(f"\n{name}")
        print("  All paired differences are ~0; Wilcoxon not defined.")
        print("  n =", len(d))
        print("  median(a-b) =", float(np.median(d)))
        return

    # Use mode='auto' for SciPy>=1.9 (falls back appropriately)
    res = wilcoxon(a, b, zero_method=zero_method, alternative=alternative, mode="auto")

    print(f"\n{name}")
    print("  n =", len(d))
    print("  median(a-b) =", float(np.median(d)))
    print("  mean(a-b)   =", float(np.mean(d)))
    print("  W =", float(res.statistic))
    print("  p =", float(res.pvalue))

paired_wilcoxon(df[col_acc_1], df[col_acc_2], "CBM1 acc vs CBM2 acc (cbm1_acc - cbm2_acc)")
paired_wilcoxon(df[col_F_1],  df[col_F_2],  "F raw pair12 mean: CBM1 vs CBM2")
paired_wilcoxon(df[col_E_1],  df[col_E_2],  "E raw pair12 mean: CBM1 vs CBM2")


CBM1 acc vs CBM2 acc (cbm1_acc - cbm2_acc)
  n = 10
  median(a-b) = -0.12921348314606734
  mean(a-b)   = -0.11555892125438812
  W = 8.0
  p = 0.09765625

F raw pair12 mean: CBM1 vs CBM2
  n = 10
  median(a-b) = 0.01841194715234445
  mean(a-b)   = 0.06143174763492424
  W = 8.0
  p = 0.048828125

E raw pair12 mean: CBM1 vs CBM2
  n = 10
  median(a-b) = 0.0002489029429852503
  mean(a-b)   = 0.0002394847571849797
  W = 21.0
  p = 0.556640625
